# AMADS Coding Notebooks

## Syncopation Span

Exploratory notebook for `amads.time.meter.{representations.py, syncopation_span.py}`.

Covers:
1. Building metrical structures (three input routes)
2. Inspecting weight maps and coincidence lists
3. Transition heatmaps — all within-cycle onset pairs
4. Onset-sequence analysis with `analyse()`

---

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key:**
- <font color='green'> Green is for a block of information.
- <font color='purple'> Purple is for an exercise.
- <font color='crimson'> Crimson is for the solution to the exercise above it.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from amads.time.meter.representations import (
    TimeSignature,
    PulseLengths,
    StartTimeHierarchy,
    VariableNesting,
)
from amads.time.meter.syncopation_span import (
    _build_weight_map,
    plot_transition_heatmap,
    analyse,
    default_decay,
)

---
## <font color='green'> Many Ways to Build Metrical Structures

A `StartTimeHierarchy` represents time points in a metrical cycle and their accentual status,
organising the data by explicitly delimited levels.

Here are three routes to forming a `StartTimeHierarchy`:

| Example | Route                               | Best for                                              |
|---------|-------------------------------------|-------------------------------------------------------|
| 1       | `TimeSignature(as_string=…)`        | Standard Western meters                               |
| 2       | `PulseLengths([…], cycle_length=…)` | Any fully isochronous hierarchy, explicit control     |
| 3       | `VariableNesting([…])`              | Non-isochronous / mixed subdivision (e.g. Jeongganbo) |

We will set up these options now and re-use several of the structure as recurring examples

### Example 1: Time-signature string

Note the `fill_2s_3s()` to add the conventional intermediate levels in 2- and 3- based metrical groupings
(e.g. half-bar in 4/4)

In [ ]:
ts_4_4 = TimeSignature(as_string='4/4')
ts_4_4.fill_2s_3s()
sh_4_4 = StartTimeHierarchy(ts_4_4.to_start_hierarchy())
print('4/4  pulse levels:', sh_4_4.compute_pulse_lengths())

ts_6_8 = TimeSignature(as_string='6/8')
ts_6_8.fill_2s_3s()
sh_6_8 = StartTimeHierarchy(ts_6_8.to_start_hierarchy())
print('6/8  pulse levels:', sh_6_8.compute_pulse_lengths())

ts_3_4 = TimeSignature(as_string='3/4')
ts_3_4.fill_2s_3s()
sh_3_4 = StartTimeHierarchy(ts_3_4.to_start_hierarchy())
print('3/4  pulse levels:', sh_3_4.compute_pulse_lengths())

# Asymmetric: 2+3/4 — beat layer is explicit in the hierarchy
ts_2_3 = TimeSignature(as_string='2+3/4')
sh_2_3 = StartTimeHierarchy(ts_2_3.to_start_hierarchy())
print('2+3/4 hierarchy:', sh_2_3.start_hierarchy)

### Example 2: Explicit pulse lengths

For precise control over which levels exist.

In [ ]:
pl_binary = PulseLengths([4, 2, 1, 0.5], cycle_length=4)
sh_binary = StartTimeHierarchy(pl_binary.to_start_hierarchy())
print('binary 4-level hierarchy:')
for level in sh_binary.start_hierarchy:
    print(' ', level)

# A 12/8-style hierarchy built from pulse lengths
pl_12_8 = PulseLengths([6, 3, 1.5, 0.5], cycle_length=6)
sh_12_8 = StartTimeHierarchy(pl_12_8.to_start_hierarchy())
print('\n12/8-style hierarchy:')
for level in sh_12_8.start_hierarchy:
    print(' ', level)

 ### Example 3: Variable Nesting (non-isochronous)

Sub-lists introduce local subdivision at that position only.

Useful for rhythmic systems where subdivision varies per beat.

In [ ]:
# Simple example: a 4-beat bar where beats 1 and 3 are subdivided
vn_simple = VariableNesting([[0, 0.5, 1], 2, [3, 3.5, 4]])
sh_vn_simple = StartTimeHierarchy(vn_simple.to_start_hierarchy())
print('Mixed subdivision:')
for level in sh_vn_simple.start_hierarchy:
    print(' ', level)

# Jeongganbo-style (from module docstring example and the book)
jg = [[0, 4, 8], 12, 24, 36, [48, 52, 56], 60, 72, [84, [88, 90], 92], 96, [108, 112, 116], 120]
sh_jg = StartTimeHierarchy(VariableNesting(jg).to_start_hierarchy())
print('\nJeongganbo — number of levels:', len(sh_jg.start_hierarchy))
print('cycle length:', sh_jg.cycle_length)

---

## <font color='green'> Weight maps and coincidence lists

A *weight* at a position can be as simple as the
number of pulse streams coinciding there — i.e. the coincident pulse count from
`coincident_pulse_list`.

In [ ]:
def show_weights(sh, granular_pulse, title=''):
    """Print position and weight mapping for a hierarchy."""
    wm = _build_weight_map(sh, granular_pulse)
    positions = sorted(wm)
    weights   = [wm[p] for p in positions]
    print(f'{title}  (granular_pulse={granular_pulse})')
    for p, w in zip(positions, weights):
        bar = '█' * w
        print(f'  {p:5.2f}  {bar}  ({w})')
    print()

show_weights(sh_4_4, 0.5, '4/4 with half-bar level')
show_weights(sh_6_8, 0.5, '6/8 with dotted-quarter level')
show_weights(sh_2_3, 1.0, '2+3/4 asymmetric')

In [ ]:
# Visualise weight profiles as a bar chart

import matplotlib.ticker as ticker # import here as it's used here (once) only

def plot_weight_profile(sh, granular_pulse, title, ax):
    wm = _build_weight_map(sh, granular_pulse)
    positions = sorted(wm)
    weights   = [wm[p] for p in positions]
    colors    = plt.cm.YlOrRd(np.array(weights) / max(weights))
    ax.bar(
        positions, weights, width=granular_pulse*0.8, color=colors,
        align='edge', edgecolor='white',  # separator (assuming white BG)
        linewidth=0.5
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Position (quarter lengths)')
    ax.set_ylabel('Weight')
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.set_xlim(0, sh.cycle_length)

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
plot_weight_profile(sh_4_4, 0.5,  '4/4 (eighth-note grid)',         axes[0,0])
plot_weight_profile(sh_6_8, 0.5,  '6/8 (eighth-note grid)',         axes[0,1])
plot_weight_profile(sh_2_3, 1.0,  '2+3/4 asymmetric (quarter grid)', axes[1,0])
plot_weight_profile(sh_3_4, 0.5,  '3/4 (eighth-note grid)',         axes[1,1])
plt.suptitle('Metrical weight profiles', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## <font color='green'> Transition heatmaps

For every ordered pair of grid positions (A, B) within one cycle, the score is the syncopation value of a note starting at A and resolving at B:

$$\text{score}(A,B) = \text{count} \times \text{gap}$$

where *count* is the number of strictly higher-weight positions in the open interval $(A, B)$ and *gap* is the difference between the maximum weight traversed and the weight at A. The lower triangle is masked (B ≤ A is impossible).

In [ ]:
# Standard meters side by side 

plot_transition_heatmap(
    sh_4_4, 0.5, mask_lower=False, title='4/4  (eighth-note grid)')
plot_transition_heatmap(
    sh_6_8, 0.5, mask_lower=False, title='6/8  (eighth-note grid)')
plot_transition_heatmap(
    sh_3_4, 0.5, mask_lower=False, title='3/4  (eighth-note grid)')

In [ ]:
# Asymmetric meter and custom pulse hierarchy

plot_transition_heatmap(sh_2_3,    1.0, mask_lower=True,
                        title='2+3/4  (quarter-note grid)')
plot_transition_heatmap(sh_binary, 0.5, mask_lower=True,
                        title='4-level binary (PulseLengths)')

Now see the effect of `fill_2s_3s` in the case of the 4/4 with vs without the half-bar level.

In [ ]:
# no fill: only /1 and /4
ts_4_4_thin = TimeSignature(as_string='4/4')
sh_thin = StartTimeHierarchy(ts_4_4_thin.to_start_hierarchy())

# fill: /1, /2, and /4
ts_4_4_full = TimeSignature(as_string='4/4')
ts_4_4_full.fill_2s_3s()
sh_full = StartTimeHierarchy(ts_4_4_full.to_start_hierarchy())

plot_transition_heatmap(sh_thin, 0.5, mask_lower=True,
                        title='4/4  without fill_2s_3s  (/1 and /4 only)')
plot_transition_heatmap(sh_full, 0.5, mask_lower=True,
                        title='4/4  with fill_2s_3s  (/1, /2, /4)')

Now let's explore custom `level_weights`.
- The default is a simple rank (1 per level).
- Any custom can be applied, for instance, the following example level doubles the value per level.

In [ ]:
exp_weights = [1, 2, 4, 8]

plot_transition_heatmap(sh_4_4, 0.5, mask_lower=True,
                        title='4/4  default rank weights')
plot_transition_heatmap(sh_4_4, 0.5, level_weights=exp_weights, mask_lower=True,
                        title='4/4  exponential level weights')

---
## <font color='green'>  Onset-sequence analysis with `analyse()`

`analyse()` takes a concrete list of onset positions and scores each one given the next `max_lookahead` onsets.
The decay function determines how much weight each successive lookahead pair receives (default: `1/n`).

### Example 1: Minimal, with four quarter notes in 4/4

In [ ]:
onsets_straight = [0.0, 1.0, 2.0, 3.0]

result = analyse(
    onsets=onsets_straight,
    hierarchy=sh_4_4,
    granular_pulse=0.5,
    max_lookahead=2,
)

print(f'Total syncopation score: {result.total_score}\n')
for os in result.onset_results:
    print(f'  onset {os.onset:.2f}  weight={os.weight}  score={os.score:.2f}')
    for pair in os.pairs:
        print(f'... resolves at {pair.onset_b:.2f}  '
              f'gap={pair.gap}  count={pair.count}  '
              f'decay={pair.decay:.2f}  pair_score={pair.score:.2f}')

### Example 2: Syncopated rhythm

Off-beat eighth notes.
Onset at 0.5 sustains across the quarter-beat at 1.0 (higher weight).

In [ ]:
onsets_sync = [0.0, 0.5, 1.5, 2.0, 2.5, 3.5]

result_sync = analyse(
    onsets=onsets_sync,
    hierarchy=sh_4_4,
    granular_pulse=0.5,
    max_lookahead=2,
)

print(f'Total syncopation score (syncopated rhythm): {result_sync.total_score}\n')
for os in result_sync.onset_results:
    flag = '  syncopated' if os.score > 0 else ''
    print(f'  onset {os.onset:.2f}  weight={os.weight}  score={os.score:.2f}{flag}')

In [ ]:
# Visualise per-onset scores

def plot_onset_scores(result, sh, granular_pulse, title, ax):
    wm = _build_weight_map(sh, granular_pulse)
    onsets = [os.onset for os in result.onset_results]
    scores = [os.score for os in result.onset_results]

    # background: weight profile of the full grid
    all_pos = sorted(wm)
    all_w   = [wm[p] for p in all_pos]
    ax.bar(all_pos, all_w, width=granular_pulse*0.9,
           align='edge', color='#e8e4d8', label='Grid weight')

    # syncopation scores
    sc_colors = ['#b01e00' if s > 0 else '#888' for s in scores]
    ax.bar(onsets, scores, width=granular_pulse*0.6,
           align='edge', color=sc_colors, alpha=0.85, label='Sync score')

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Position (quarter lengths)')
    ax.set_ylabel('Value')
    ax.legend(fontsize=8)
    ax.set_xlim(-0.1, sh.cycle_length + 0.1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_onset_scores(result,      sh_4_4, 0.5, 'Straight quarter notes', axes[0])
plot_onset_scores(result_sync, sh_4_4, 0.5, 'Syncopated rhythm',       axes[1])
plt.suptitle('Per-onset syncopation scores (red) against weight profile (grey)',
             fontsize=12)
plt.tight_layout()
plt.show()

### Example 3: Cycle wrapping with onset near the end of the bar

An onset at position 3.5 in a 4/4 bar sustains across the downbeat
(past position 0 of the next cycle, which has the highest metrical weight).

The cycle wraps automatically.

In [ ]:
onsets_wrap = [0.0, 1.0, 2.0, 3.5, 4.5]

result_wrap = analyse(
    onsets=onsets_wrap,
    hierarchy=sh_4_4,
    granular_pulse=0.5,
    max_lookahead=1,
)

print('Onsets crossing the barline:')
for os in result_wrap.onset_results:
    flag = '  syncopated (crosses barline)' if os.score > 0 else ''
    print(f'  onset {os.onset:.2f}  weight={os.weight}  score={os.score:.2f}{flag}')

Custom decay functions

Example of 'hard cutoff' where only the immediate successor counts.

In [ ]:
def hard_cutoff(n):
    return 1.0 if n == 1 else 0.0

result_hard = analyse(
    onsets=onsets_sync,
    hierarchy=sh_4_4,
    granular_pulse=0.5,
    decay_fn=hard_cutoff,
    max_lookahead=3,
)

result_soft = analyse(
    onsets=onsets_sync,
    hierarchy=sh_4_4,
    granular_pulse=0.5,
    decay_fn=default_decay,   # 1/n
    max_lookahead=3,
)

print('Decay comparison (same rhythm, max_lookahead=3):')
print(f'{"onset":>8}  {"hard_cutoff":>12}  {"default_1/n":>12}')
for h, s in zip(result_hard.onset_results, result_soft.onset_results):
    print(f'{h.onset:>8.2f}  {h.score:>12.2f}  {s.score:>12.2f}')
print(f'{"total":>8}  {result_hard.total_score:>12.2f}  {result_soft.total_score:>12.2f}')

---
## <font color='green'> Heatmap for a Korean Jeongganbo Meter

The `VariableNesting` route allows exploration of meters that cannot be expressed as a flat list of isochronous pulse lengths.

In [ ]:
# Check we still have the JG example:
sh_jg.start_hierarchy[1]

In [ ]:
print('Jeongganbo hierarchy levels:', len(sh_jg.start_hierarchy))
for level in sh_jg.start_hierarchy:
    print(' ', level)

plot_transition_heatmap(
    sh_jg, 4.0, mask_lower=True,
    title='Jeongganbo-style (granular_pulse=4)'
)

End

---